In [1]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot  as plt
import numpy as np

In [2]:
train_df = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/train.csv')
test_df = pd.read_csv('/kaggle/input/competitions/house-prices-advanced-regression-techniques/test.csv')

In [3]:
train_df.columns

Index(['Id', 'MSSubClass', 'MSZoning', 'LotFrontage', 'LotArea', 'Street',
       'Alley', 'LotShape', 'LandContour', 'Utilities', 'LotConfig',
       'LandSlope', 'Neighborhood', 'Condition1', 'Condition2', 'BldgType',
       'HouseStyle', 'OverallQual', 'OverallCond', 'YearBuilt', 'YearRemodAdd',
       'RoofStyle', 'RoofMatl', 'Exterior1st', 'Exterior2nd', 'MasVnrType',
       'MasVnrArea', 'ExterQual', 'ExterCond', 'Foundation', 'BsmtQual',
       'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1',
       'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Heating',
       'HeatingQC', 'CentralAir', 'Electrical', '1stFlrSF', '2ndFlrSF',
       'LowQualFinSF', 'GrLivArea', 'BsmtFullBath', 'BsmtHalfBath', 'FullBath',
       'HalfBath', 'BedroomAbvGr', 'KitchenAbvGr', 'KitchenQual',
       'TotRmsAbvGrd', 'Functional', 'Fireplaces', 'FireplaceQu', 'GarageType',
       'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual',
       'GarageCond', 'PavedDrive

In [4]:
train_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1460 entries, 0 to 1459
Data columns (total 81 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   Id             1460 non-null   int64  
 1   MSSubClass     1460 non-null   int64  
 2   MSZoning       1460 non-null   object 
 3   LotFrontage    1201 non-null   float64
 4   LotArea        1460 non-null   int64  
 5   Street         1460 non-null   object 
 6   Alley          91 non-null     object 
 7   LotShape       1460 non-null   object 
 8   LandContour    1460 non-null   object 
 9   Utilities      1460 non-null   object 
 10  LotConfig      1460 non-null   object 
 11  LandSlope      1460 non-null   object 
 12  Neighborhood   1460 non-null   object 
 13  Condition1     1460 non-null   object 
 14  Condition2     1460 non-null   object 
 15  BldgType       1460 non-null   object 
 16  HouseStyle     1460 non-null   object 
 17  OverallQual    1460 non-null   int64  
 18  OverallC

### Column Meaning


# Feature Engineering (Dtype transformation, Feature Transformation, Null Handling, Remove/Add New features)

## Dtype Transformation

Since `MSSubClass` is suppose to be an `object` class, not `int`, we will convert it

In [5]:
for df in [train_df, test_df]:
    df['MSSubClass'] = df['MSSubClass'].astype(str)

we wil construct the mapping of the ordinal feature, based on the description of the dataset

In [6]:
ORDINAL_MAPS = {
    'ExterQual'   : {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'ExterCond'   : {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'BsmtQual'    : {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'None':0},
    'BsmtCond'    : {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'None':0},
    'HeatingQC'   : {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'KitchenQual' : {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1},
    'FireplaceQu' : {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'None':0},
    'GarageQual'  : {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'None':0},
    'GarageCond'  : {'Ex':5, 'Gd':4, 'TA':3, 'Fa':2, 'Po':1, 'None':0},
    'PoolQC'      : {'Ex':4, 'Gd':3, 'TA':2, 'Fa':1, 'None':0},
    # Basement exposure: Gd > Av > Mn > No > NA
    'BsmtExposure': {'Gd':4, 'Av':3, 'Mn':2, 'No':1, 'None':0},
    # Basement finish type: GLQ > ALQ > BLQ > Rec > LwQ > Unf > NA
    'BsmtFinType1': {'GLQ':6, 'ALQ':5, 'BLQ':4, 'Rec':3, 'LwQ':2, 'Unf':1, 'None':0},
    'BsmtFinType2': {'GLQ':6, 'ALQ':5, 'BLQ':4, 'Rec':3, 'LwQ':2, 'Unf':1, 'None':0},
    # Garage finish: Fin > RFn > Unf > NA
    'GarageFinish': {'Fin':3, 'RFn':2, 'Unf':1, 'None':0},
    # Functional: Typ (best) ... Sal (worst)
    'Functional'  : {'Typ':7, 'Min1':6, 'Min2':5, 'Mod':4,
                     'Maj1':3, 'Maj2':2, 'Sev':1, 'Sal':0},
    # Fence: GdPrv > MnPrv > GdWo > MnWw > NA
    'Fence'       : {'GdPrv':4, 'MnPrv':3, 'GdWo':2, 'MnWw':1, 'None':0},
    # Lot shape: Reg (most regular) ... IR3 (most irregular)
    'LotShape'    : {'Reg':3, 'IR1':2, 'IR2':1, 'IR3':0},
    # Land slope
    'LandSlope'   : {'Gtl':2, 'Mod':1, 'Sev':0},
    # Paved drive
    'PavedDrive'  : {'Y':2, 'P':1, 'N':0},
}

In [7]:
train_ids = train_df['Id']
test_ids = test_df['Id']

train_df.drop('Id', axis=1, inplace=True)
test_df.drop('Id', axis=1, inplace=True)

In [8]:
outlier_mask = (
    (train_df['GrLivArea'] > 4000) & (train_df['SalePrice'] < 300_000)
)
n_outliers = outlier_mask.sum()

train_df = train_df[~outlier_mask].reset_index(drop=True)

In [9]:
y_train = train_df['SalePrice']
train_df.drop('SalePrice', axis=1, inplace=True)

In [10]:
all_data = pd.concat([train_df, test_df], axis=0, ignore_index=True)
all_data.shape

(2917, 79)

## Duplicate Handling

In [11]:
feature_cols = [c for c in train_df.columns if c not in ['Id', 'SalePrice']]
print(f'Duplicate rows in train: {train_df.duplicated().sum()}')

Duplicate rows in train: 0


In [12]:
constant = [c for c in feature_cols if all_data[c].nunique() <= 1]
print(f'Constant features: {constant if constant else "None"}')


Constant features: None


In [13]:
print('\nNear-constant features (>=98% one value):')
cols_to_drop = []
for col in feature_cols:
    vc = all_data[col].dropna().value_counts(normalize=True)
    if len(vc) >= 1 and vc.iloc[0] >= 0.98:
        print(f'  {col:<22} dominant="{vc.index[0]}"  ({vc.iloc[0]*100:.1f}%)')
        cols_to_drop.append(col)



Near-constant features (>=98% one value):
  Street                 dominant="Pave"  (99.6%)
  Utilities              dominant="AllPub"  (100.0%)
  Condition2             dominant="Norm"  (99.0%)
  RoofMatl               dominant="CompShg"  (98.6%)
  Heating                dominant="GasA"  (98.5%)
  LowQualFinSF           dominant="0"  (98.6%)
  3SsnPorch              dominant="0"  (98.7%)
  PoolArea               dominant="0"  (99.6%)


In [14]:
for col in cols_to_drop:
    all_data.drop(col, axis=1, inplace=True)
    print(f"{col=} is dropped")


col='Street' is dropped
col='Utilities' is dropped
col='Condition2' is dropped
col='RoofMatl' is dropped
col='Heating' is dropped
col='LowQualFinSF' is dropped
col='3SsnPorch' is dropped
col='PoolArea' is dropped


We will remove some of the near-constant value later on, but now we will remove the `Id` column

## Null Handling

In [15]:
NONE_COLS = all_data.columns[all_data.isnull().any()].tolist()
print(NONE_COLS)

['MSZoning', 'LotFrontage', 'Alley', 'Exterior1st', 'Exterior2nd', 'MasVnrType', 'MasVnrArea', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinSF1', 'BsmtFinType2', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF', 'Electrical', 'BsmtFullBath', 'BsmtHalfBath', 'KitchenQual', 'Functional', 'FireplaceQu', 'GarageType', 'GarageYrBlt', 'GarageFinish', 'GarageCars', 'GarageArea', 'GarageQual', 'GarageCond', 'PoolQC', 'Fence', 'MiscFeature', 'SaleType']


In [16]:
miss = all_data.isnull().sum()
miss = miss[miss > 0].sort_values(ascending=False)
miss_pct = (miss / len(all_data) * 100).round(2)
miss_df = pd.DataFrame({
    'missing_count'      : miss,
    'missing_pct'        : miss_pct,
})
miss_df

,missing_count,missing_pct
PoolQC,2908,99.69
MiscFeature,2812,96.40
Alley,2719,93.21
Fence,2346,80.43
MasVnrType,1766,60.54
FireplaceQu,1420,48.68
LotFrontage,486,16.66
GarageQual,159,5.45
GarageCond,159,5.45
GarageYrBlt,159,5.45


In [17]:
# --- 4a. NaN = ABSENCE -> fill with 'None' (categorical) ---
none_categoricals = [
    'Alley', 'BsmtQual', 'BsmtCond', 'BsmtExposure',
    'BsmtFinType1', 'BsmtFinType2', 'FireplaceQu',
    'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond',
    'PoolQC', 'Fence', 'MiscFeature', 'MasVnrType'
]

for col in none_categoricals:
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna('None')

In [18]:
# --- 4b. NaN = ABSENCE -> fill with 0 (numeric) ---
zero_numerics = [
    'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', 'TotalBsmtSF',
    'BsmtFullBath', 'BsmtHalfBath',
    'GarageArea', 'GarageCars',
    'MasVnrArea',
    'PoolArea',
    'MiscVal',
]

for col in zero_numerics:
    if col in all_data.columns:
        all_data[col] = all_data[col].fillna(0)

In [19]:
#  4c. GarageYrBlt: NaN = no garage -> impute with YearBuilt ---
all_data['GarageYrBlt'] = all_data['GarageYrBlt'].fillna(all_data['YearBuilt'])

#  4d. LotFrontage: NaN -> impute with neighborhood median ---
all_data['LotFrontage'] = all_data.groupby('Neighborhood')['LotFrontage'].transform(
    lambda x: x.fillna(x.median())
)

In [20]:
# import seaborn as sns
# import matplotlib.pyplot as plt
# import pandas as pd

# # 1. Heatmap: Shows the concentration of counts
# plt.figure(figsize=(10, 8))
# ct = pd.crosstab(all_data['MSSubClass'], all_data['MSZoning'])
# sns.heatmap(ct, annot=True, fmt='d', cmap='YlGnBu')
# plt.title('Relationship between MSSubClass and MSZoning (Frequency Count)')
# plt.show()

# # 2. Stacked Bar Chart: Shows the percentage distribution
# # This helps see which Zoning is the "Mode" for each SubClass
# ct_pct = pd.crosstab(all_data['MSSubClass'], all_data['MSZoning'], normalize='index')
# ct_pct.plot(kind='bar', stacked=True, figsize=(12, 6), colormap='viridis')
# plt.title('MSZoning Distribution by MSSubClass (Normalized)')
# plt.ylabel('Percentage')
# plt.legend(title='MSZoning', bbox_to_anchor=(1.05, 1), loc='upper left')
# plt.tight_layout()
# plt.show()

In [21]:
# --- 4e. MSZoning: NaN -> impute with mode per MSSubClass ---
all_data['MSZoning'] = all_data.groupby('MSSubClass')['MSZoning'].transform(
    lambda x: x.fillna(x.mode()[0])
)


In [22]:
cat_cols = all_data.select_dtypes(include=['object']).columns
for col in cat_cols:
    if all_data[col].isnull().sum() > 0:
        all_data[col] = all_data[col].fillna(all_data[col].mode()[0])


In [23]:
# --- 4h. Remaining numeric NaN -> median ---
num_cols = all_data.select_dtypes(include=[np.number]).columns
for col in num_cols:
    if all_data[col].isnull().sum() > 0:
        all_data[col] = all_data[col].fillna(all_data[col].median())

print(f"Missing values remaining: {all_data.isnull().sum().sum()}")

Missing values remaining: 0


In [24]:
# for col, mapping in ORDINAL_MAPS.items():
#     if col in all_data.columns:
#         print(col)
#         all_data[col] = all_data[col].map(mapping)

In [25]:
print(f"Missing values remaining: {all_data.isnull().sum().sum()}")

Missing values remaining: 0


In [26]:
nominal_cols = [
    c for c in all_data.select_dtypes(include=['object']).columns if c not in ORDINAL_MAPS
]
for col in nominal_cols:
    vc = all_data[col].value_counts(normalize=True)
    rare_cats = vc[vc < 0.01].index
    if len(rare_cats) > 0:
        all_data[col] = all_data[col].replace(rare_cats, "Others")
        print(f"{col=} grouped {len(rare_cats)=} -> Others")
        

col='MSSubClass' grouped len(rare_cats)=5 -> Others
col='MSZoning' grouped len(rare_cats)=2 -> Others
col='LotConfig' grouped len(rare_cats)=1 -> Others
col='Neighborhood' grouped len(rare_cats)=4 -> Others
col='Condition1' grouped len(rare_cats)=4 -> Others
col='HouseStyle' grouped len(rare_cats)=3 -> Others
col='RoofStyle' grouped len(rare_cats)=4 -> Others
col='Exterior1st' grouped len(rare_cats)=5 -> Others
col='Exterior2nd' grouped len(rare_cats)=6 -> Others
col='MasVnrType' grouped len(rare_cats)=1 -> Others
col='Foundation' grouped len(rare_cats)=2 -> Others
col='Electrical' grouped len(rare_cats)=2 -> Others
col='GarageType' grouped len(rare_cats)=2 -> Others
col='MiscFeature' grouped len(rare_cats)=3 -> Others
col='SaleType' grouped len(rare_cats)=6 -> Others
col='SaleCondition' grouped len(rare_cats)=2 -> Others


## Add new feature

#### HouseAge = YrSold - YearBuilt
- Yearbuilt/YearSold is an absulute calendar year. A house built in 1960 sold in 2006 as 46 years old. The age of time of sale matters for price, not the calendar year it was built
- Older houses tend to sell for less (unless it is rennovated, in the following section) -> age is the meaningful dimenstion

#### RemodAge = YrSold - YearRemodAdd
- A house built-in 1960 but remodeled in 2005 -> newer than house built in 1960 but never remodeled

#### GarageAge
- Maybe the price can be related based on the age of the garage. Age == 0 -> garage integrated, Age < 0 => garage added later, maybe more expensive?

In [27]:
all_data['HouseAge'] = all_data['YrSold'] - all_data['YearBuilt']
all_data['RemodAge'] = all_data['YrSold'] - all_data['YearRemodAdd']
all_data['GarageAge'] = all_data['YrSold'] - all_data['GarageYrBlt']
all_data['YearsSinceRemod'] = all_data['YearRemodAdd'] - all_data['YearBuilt']
all_data['YearsSinceRemod'] = all_data['YearsSinceRemod'].clip(lower=0)

In [28]:
# all_data.drop(columns=['YrSold', 'YearBuilt', 'YearRemodAdd', 'GarageYrBlt'], inplace=True, errors='ignore')

#### TotalSF = basemanet + 1st floor + 2nd floor
Normally, we calculate the total space of the house, not individually, when it comes to the price of the house

#### TotalPorchSF
These area is not that common -> quite sparse. It is easier for the model just to learn the general `outdoor` commonity.

In [29]:
all_data['TotalSF'] = (
    all_data['TotalBsmtSF'] +
    all_data['1stFlrSF'] +
    all_data['2ndFlrSF']
)

all_data['TotalPorchSF'] = (  # remove sparsity from these columns
    all_data['WoodDeckSF'] +
    all_data['OpenPorchSF'] +
    all_data['EnclosedPorch'] +
    all_data['ScreenPorch']
)

In [30]:
# all_data.drop(columns=['ScreenPorch', 'TotalPorchSF'], inplace=True, errors='ignore')

#### TotalBath
Same pattern as above feature. 
It is more easy for us, and the model to interpret total number of bath rooms. For the HalfBath, since it is just half -> 0.5 weight.  And not every house has Basement Bathroom -> remove the sparsity 

In [31]:
all_data['TotalBath'] = (
    all_data['FullBath'] +
    0.5 * all_data['HalfBath'] +
    all_data['BsmtFullBath'] +
    0.5 * all_data['BsmtHalfBath']
)

In [32]:
# all_data.drop(columns=['HalfBath', 'BsmtHalfBath'], inplace=True, errors='ignore')

#### Binary-Alike Feature -> Binary Flag
These transformation serve like a threshold effects

In [33]:
all_data['HasGarage'] = (all_data['GarageArea'] > 0).astype(object)
all_data['HasFireplace'] = (all_data['Fireplaces'] > 0).astype(object)
all_data['Has2ndFloor']  = (all_data['2ndFlrSF'] > 0).astype(object)
all_data['HasBsmt'] = (all_data['TotalBsmtSF'] > 0).astype(object)
all_data['HasWoodDeck'] = (all_data['WoodDeckSF'] > 0).astype(object)

In [34]:
all_data['LivingLotRatio'] = all_data['GrLivArea'] / all_data['LotArea']
all_data['BsmtRatio'] = all_data['TotalBsmtSF'] / all_data['TotalSF'].clip(lower=1)

In [36]:
feature_cols = [c for c in all_data.columns if c not in ['Id', 'SalePrice']]
print('\nNear-constant features (>=95% one value):')
cols_to_drop_98 = []
for col in feature_cols:
    vc = all_data[col].dropna().value_counts(normalize=True)
    if len(vc) >= 1 and vc.iloc[0] >= 0.98:
        print(f'  {col:<22} dominant="{vc.index[0]}"  ({vc.iloc[0]*100:.1f}%)')
        cols_to_drop_98.append(col)



Near-constant features (>=95% one value):
  PoolQC                 dominant="None"  (99.7%)


In [39]:
all_data.drop(columns=cols_to_drop_98, inplace=True, errors='ignore')

print(f"Remaining columns: {all_data.shape[1]}")

Remaining columns: 84


In [40]:
# From the visualization, a lot of numerical feature is skew -> log trasnformation

import numpy as np

feature_cols = [c for c in all_data.columns if c not in ['Id', 'SalePrice']]
numeric_cols = all_data[feature_cols].select_dtypes(include=[np.number]).columns.tolist()

skew_threshold = 0.75
skew_vals = all_data[numeric_cols].skew()

highly_skewed = skew_vals[skew_vals.abs() > skew_threshold].index.tolist()

print(f"\nApplying log1p to {len(highly_skewed)} highly skewed features (|skew| > {skew_threshold}):")

for col in highly_skewed:
    min_val = all_data[col].min()
    if min_val < 0:
        all_data[col] = all_data[col] - min_val

    all_data[col] = np.log1p(all_data[col])

    print(f"{col}: skew {skew_vals[col]:.2f} -> {all_data[col].skew():.2f}")


Applying log1p to 12 highly skewed features (|skew| > 0.75):
LotFrontage: skew -1.07 -> -1.42
BsmtFinSF2: skew 2.46 -> 2.38
BsmtUnfSF: skew -2.16 -> -2.79
BsmtHalfBath: skew 3.78 -> 3.74
KitchenAbvGr: skew 3.52 -> 2.24
EnclosedPorch: skew 1.96 -> 1.90
ScreenPorch: skew 2.95 -> 2.92
MiscVal: skew 5.21 -> 5.07
YearsSinceRemod: skew 0.94 -> 0.68
TotalPorchSF: skew -1.34 -> -1.65
LivingLotRatio: skew 2.34 -> 1.99
BsmtRatio: skew -1.86 -> -2.24


In [41]:
col_to_check_near_constant = ['MasVnrArea', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF', '2ndFlrSF',  'WoodDeckSF', 'OpenPorchSF', 'EnclosedPorch']

for col in col_to_check_near_constant:
    vc = all_data[col].dropna().value_counts(normalize=True)
    if len(vc) >= 1 and vc.iloc[0] >= 0.1:
        print(f'  {col:<22} dominant="{vc.index[0]}"  ({vc.iloc[0]*100:.1f}%)')


  MasVnrArea             dominant="0.0"  (60.4%)
  BsmtFinSF1             dominant="0.0"  (31.9%)
  BsmtFinSF2             dominant="0.0"  (88.1%)
  2ndFlrSF               dominant="0.0"  (57.2%)
  WoodDeckSF             dominant="0.0"  (52.2%)
  OpenPorchSF            dominant="0.0"  (44.5%)
  EnclosedPorch          dominant="0.0"  (84.3%)


In [42]:
for col, mapping in ORDINAL_MAPS.items():
    if col in all_data.columns:
        print(col)
        all_data[col] = all_data[col].map(mapping)

ExterQual
ExterCond
BsmtQual
BsmtCond
HeatingQC
KitchenQual
FireplaceQu
GarageQual
GarageCond
BsmtExposure
BsmtFinType1
BsmtFinType2
GarageFinish
Functional
Fence
LotShape
LandSlope
PavedDrive


In [43]:
# ============================================================
# TARGET TRANSFORMATION + TARGET ENCODING
# ============================================================

y_train = np.log1p(y_train)
print(f'Target skew after log1p: {pd.Series(y_train).skew():.4f}')

n_train = len(y_train)
train_temp = all_data.iloc[:n_train].copy()
test_temp  = all_data.iloc[n_train:].copy()

high_card = ['Neighborhood', 'Exterior1st', 'Exterior2nd', 'SaleType']
for col in high_card:
    if col in train_temp.columns:
        means = train_temp.groupby(col).apply(lambda g: y_train.loc[g.index].mean())
        grand_mean = y_train.mean()
        counts = train_temp[col].value_counts()
        smoothing = 10  
        blend = (counts * means + smoothing * grand_mean) / (counts + smoothing)

        train_temp[col + '_tgt'] = train_temp[col].map(blend).fillna(grand_mean)
        test_temp[col + '_tgt']  = test_temp[col].map(blend).fillna(grand_mean)

        train_temp.drop(col, axis=1, inplace=True)
        test_temp.drop(col,  axis=1, inplace=True)
        print(f'  Target-encoded: {col} -> {col}_tgt')

print(f'\nTrain shape: {train_temp.shape} | Test shape: {test_temp.shape}')

Target skew after log1p: 0.1216
  Target-encoded: Neighborhood -> Neighborhood_tgt
  Target-encoded: Exterior1st -> Exterior1st_tgt
  Target-encoded: Exterior2nd -> Exterior2nd_tgt
  Target-encoded: SaleType -> SaleType_tgt

Train shape: (1458, 84) | Test shape: (1459, 84)


/tmp/ipykernel_55/2460617357.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  means = train_temp.groupby(col).apply(lambda g: y_train.loc[g.index].mean())
/tmp/ipykernel_55/2460617357.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  means = train_temp.groupby(col).apply(lambda g: y_train.loc[g.index].mean())
/tmp/ipykernel_55/2460617357.py:15: FutureWarning: DataFrameGroupBy.apply operated on the grouping col

In [44]:
# ============================================================
# ONE-HOT ENCODE REMAINING NOMINALS + ALIGN TRAIN/TEST
# ============================================================

remaining_cat = train_temp.select_dtypes(include=['object']).columns.tolist()
print(f'Remaining categorical columns to encode: {remaining_cat}')

train_encoded = pd.get_dummies(train_temp, drop_first=False)
test_encoded  = pd.get_dummies(test_temp,  drop_first=False)

train_encoded, test_encoded = train_encoded.align(test_encoded, join='left', axis=1, fill_value=0)

print(f'\nFinal Train: {train_encoded.shape}')
print(f'Final Test:  {test_encoded.shape}')
print(f'Target:      {y_train.shape}')
print(f'\nMissing values in train: {train_encoded.isnull().sum().sum()}')
print(f'Missing values in test:  {test_encoded.isnull().sum().sum()}')

Remaining categorical columns to encode: ['MSSubClass', 'MSZoning', 'Alley', 'LandContour', 'LotConfig', 'Condition1', 'BldgType', 'HouseStyle', 'RoofStyle', 'MasVnrType', 'Foundation', 'CentralAir', 'Electrical', 'GarageType', 'MiscFeature', 'SaleCondition', 'HasGarage', 'HasFireplace', 'Has2ndFloor', 'HasBsmt', 'HasWoodDeck']

Final Train: (1458, 150)
Final Test:  (1459, 150)
Target:      (1458,)

Missing values in train: 0
Missing values in test:  0


In [45]:
# ============================================================
# SAVE CLEANED DATA
# ============================================================

train_encoded.to_csv('train_cleaned.csv', index=False)
test_encoded.to_csv('test_cleaned.csv', index=False)
pd.Series(y_train, name='SalePrice').to_csv('y_train_cleaned.csv', index=False)

print('=' * 60)
print('DATA CLEANING & FEATURE ENGINEERING COMPLETE')
print('=' * 60)
print(f'  train_cleaned.csv  : {train_encoded.shape[0]} rows x {train_encoded.shape[1]} features')
print(f'  test_cleaned.csv   : {test_encoded.shape[0]} rows x {test_encoded.shape[1]} features')
print(f'  y_train_cleaned.csv: log1p(SalePrice) target')


DATA CLEANING & FEATURE ENGINEERING COMPLETE
  train_cleaned.csv  : 1458 rows x 150 features
  test_cleaned.csv   : 1459 rows x 150 features
  y_train_cleaned.csv: log1p(SalePrice) target


In [46]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.model_selection import cross_val_score, KFold
from sklearn.metrics import mean_squared_error
from sklearn.preprocessing import StandardScaler
import warnings
warnings.filterwarnings('ignore')

# Load cleaned data
X_train = pd.read_csv('train_cleaned.csv')
X_test  = pd.read_csv('test_cleaned.csv')
y_train = pd.read_csv('y_train_cleaned.csv').squeeze()

print(f'X_train: {X_train.shape}')
print(f'X_test:  {X_test.shape}')
print(f'y_train: {y_train.shape}')
print(f'\nMissing in train: {X_train.isnull().sum().sum()}')
print(f'Missing in test:  {X_test.isnull().sum().sum()}')

X_train: (1458, 150)
X_test:  (1459, 150)
y_train: (1458,)

Missing in train: 0
Missing in test:  0


In [47]:
def rmsle(y_true, y_pred):
    """Root Mean Squared Log Error"""
    return np.sqrt(mean_squared_error(
        np.expm1(y_true), np.expm1(y_pred)
    ))

def rmsle_cv(model, X, y, n_folds=5):
    kf = KFold(n_splits=n_folds, shuffle=True, random_state=42)
    scores = np.sqrt(-cross_val_score(
        model, X, y, scoring='neg_mean_squared_error', cv=kf
    ))
    return scores

print('RMSLE metric and CV helper defined.')

RMSLE metric and CV helper defined.


In [48]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled  = scaler.transform(X_test)

print(f'Scaled train: {X_train_scaled.shape}')
print(f'Scaled test:  {X_test_scaled.shape}')

Scaled train: (1458, 150)
Scaled test:  (1459, 150)


In [49]:
# ============================================================
# 1. LINEAR MODELS — Ridge, Lasso, ElasticNet
# Why: Multicollinearity (GarageCars/GarageArea), high dim (148 features),
#      log-transformed target makes relationships approximately linear.
# ============================================================

print('=== LINEAR MODELS (on scaled features) ===\n')

# Ridge — L2 regularization, handles multicollinearity
ridge = Ridge(alpha=15.0)
ridge_scores = rmsle_cv(ridge, X_train_scaled, y_train)
print(f'Ridge      : {ridge_scores.mean():.4f} (+/- {ridge_scores.std():.4f})')

# Lasso — L1 regularization, automatic feature selection
lasso = Lasso(alpha=0.0005, max_iter=5000)
lasso_scores = rmsle_cv(lasso, X_train_scaled, y_train)
print(f'Lasso      : {lasso_scores.mean():.4f} (+/- {lasso_scores.std():.4f})')

# ElasticNet — combines L1 + L2
elastic = ElasticNet(alpha=0.0005, l1_ratio=0.9, max_iter=5000)
elastic_scores = rmsle_cv(elastic, X_train_scaled, y_train)
print(f'ElasticNet : {elastic_scores.mean():.4f} (+/- {elastic_scores.std():.4f})')

# Fit Lasso on full training data to see feature selection
lasso.fit(X_train_scaled, y_train)
coef = pd.Series(lasso.coef_, index=X_train.columns)
n_selected = (coef != 0).sum()
print(f'\nLasso selected {n_selected} features out of {len(coef)}')
print('\nTop 10 Lasso coefficients (absolute):')
print(coef[coef != 0].abs().sort_values(ascending=False).head(10))

=== LINEAR MODELS (on scaled features) ===

Ridge      : 0.1173 (+/- 0.0080)
Lasso      : 0.1167 (+/- 0.0083)
ElasticNet : 0.1169 (+/- 0.0083)

Lasso selected 108 features out of 150

Top 10 Lasso coefficients (absolute):
GrLivArea           0.098787
OverallQual         0.066611
OverallCond         0.049875
TotalBsmtSF         0.042838
Neighborhood_tgt    0.041425
LotArea             0.036514
TotalSF             0.028996
BsmtFinSF1          0.023801
Functional          0.020337
TotalBath           0.018158
dtype: float64


In [50]:
# ============================================================
# 2. TREE MODELS — DecisionTree, RandomForest
# Why: Captures non-linear thresholds (e.g., OverallQual>=8 price jump),
#      interactions without explicit feature engineering.
# ============================================================

print('=== TREE MODELS (on raw features) ===\n')

# DecisionTree — baseline tree, constrained to reduce overfitting
dt = DecisionTreeRegressor(
    max_depth=5, min_samples_split=20,
    min_samples_leaf=10, random_state=42
)
dt_scores = rmsle_cv(dt, X_train, y_train)
print(f'DecisionTree : {dt_scores.mean():.4f} (+/- {dt_scores.std():.4f})')

rf = RandomForestRegressor(
    n_estimators=500, max_depth=12, min_samples_split=10,
    min_samples_leaf=5, random_state=42, n_jobs=-1
)
rf_scores = rmsle_cv(rf, X_train, y_train)
print(f'RandomForest : {rf_scores.mean():.4f} (+/- {rf_scores.std():.4f})')

rf.fit(X_train, y_train)
fi = pd.Series(rf.feature_importances_, index=X_train.columns)
print('\nTop 10 RF feature importances:')
print(fi.sort_values(ascending=False).head(10))

=== TREE MODELS (on raw features) ===

DecisionTree : 0.1816 (+/- 0.0121)
RandomForest : 0.1362 (+/- 0.0062)

Top 10 RF feature importances:
OverallQual         0.459325
TotalSF             0.330099
Neighborhood_tgt    0.056457
GrLivArea           0.010963
CentralAir_Y        0.008477
CentralAir_N        0.008064
GarageArea          0.007981
OverallCond         0.007952
BsmtFinSF1          0.007375
TotalBath           0.007191
dtype: float64


In [51]:
# ============================================================
# 3. GRADIENT BOOSTING — XGBoost, LightGBM, sklearn GBM
# Why: Best-in-class for tabular data. Learns residual errors
#      sequentially, captures complex interactions.
# ============================================================

print('=== GRADIENT BOOSTING MODELS ===\n')

gbr = GradientBoostingRegressor(
    n_estimators=3000, max_depth=4, learning_rate=0.05,
    subsample=0.85, min_samples_split=10, min_samples_leaf=5,
    random_state=42
)
gbr_scores = rmsle_cv(gbr, X_train, y_train)
print(f'GradientBoosting: {gbr_scores.mean():.4f} (+/- {gbr_scores.std():.4f})')

try:
    import xgboost as xgb
    xgb_model = xgb.XGBRegressor(
        n_estimators=3000, max_depth=4, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85,
        reg_alpha=0.1, reg_lambda=1.0,
        random_state=42, n_jobs=-1, verbosity=0
    )
    xgb_scores = rmsle_cv(xgb_model, X_train, y_train)
    print(f'XGBoost         : {xgb_scores.mean():.4f} (+/- {xgb_scores.std():.4f})')
    HAS_XGB = True
except ImportError:
    print('XGBoost         : not installed (pip install xgboost)')
    HAS_XGB = False

# LightGBM
try:
    import lightgbm as lgb
    lgb_model = lgb.LGBMRegressor(
        n_estimators=3000, max_depth=7, learning_rate=0.05,
        subsample=0.85, colsample_bytree=0.85,
        reg_alpha=0.1, reg_lambda=1.0,
        num_leaves=31, min_child_samples=10,
        random_state=42, n_jobs=-1, verbose=-1
    )
    lgb_scores = rmsle_cv(lgb_model, X_train, y_train)
    print(f'LightGBM        : {lgb_scores.mean():.4f} (+/- {lgb_scores.std():.4f})')
    HAS_LGB = True
except ImportError:
    print('LightGBM        : not installed (pip install lightgbm)')
    HAS_LGB = False

=== GRADIENT BOOSTING MODELS ===

GradientBoosting: 0.1202 (+/- 0.0038)
XGBoost         : 0.1197 (+/- 0.0050)
LightGBM        : 0.1226 (+/- 0.0062)


In [52]:
gbr.fit(X_train, y_train)

importances = gbr.feature_importances_
feat_imp = pd.Series(importances, index=X_train.columns)
feat_imp = feat_imp.sort_values(ascending=False)

print(feat_imp.head(20))

OverallQual         0.350659
TotalSF             0.337465
Neighborhood_tgt    0.066130
TotalBath           0.025083
KitchenQual         0.015617
OverallCond         0.015245
GrLivArea           0.014315
LotArea             0.010626
CentralAir_Y        0.010170
RemodAge            0.010022
BsmtFinSF1          0.009224
GarageCars          0.008656
CentralAir_N        0.007622
GarageArea          0.006451
1stFlrSF            0.006345
TotalPorchSF        0.005810
BsmtUnfSF           0.005650
FireplaceQu         0.005594
LivingLotRatio      0.004240
GarageFinish        0.003871
dtype: float64


In [55]:
xgb_model.fit(X_train, y_train)

importances = xgb_model.feature_importances_
feat_imp = pd.Series(importances, index=X_train.columns).sort_values(ascending=False)
feat_imp.head(20)

OverallQual              0.257205
CentralAir_Y             0.228923
TotalSF                  0.078143
CentralAir_N             0.062309
GarageCond               0.030187
KitchenQual              0.029014
KitchenAbvGr             0.028064
Neighborhood_tgt         0.023029
TotalBath                0.019816
GarageCars               0.019175
MSZoning_Others          0.011986
GarageQual               0.011944
MSSubClass_30            0.011723
ExterQual                0.008977
SaleCondition_Abnorml    0.008479
OverallCond              0.008360
BsmtQual                 0.008328
MSZoning_RM              0.008324
GarageType_Others        0.007253
FireplaceQu              0.007165
dtype: float32

In [56]:
lgb_model.fit(X_train, y_train)

feat_imp = pd.Series(
    lgb_model.feature_importances_,
    index=X_train.columns
).sort_values(ascending=False)
feat_imp.head(20)

TotalPorchSF        1220
LotArea             1172
LivingLotRatio      1161
BsmtUnfSF           1119
GarageArea           985
LotFrontage          893
TotalSF              856
GrLivArea            789
1stFlrSF             774
BsmtFinSF1           654
BsmtRatio            607
TotalBsmtSF          604
GarageYrBlt          587
MasVnrArea           577
MoSold               543
OpenPorchSF          540
Neighborhood_tgt     515
YearBuilt            489
WoodDeckSF           485
YearRemodAdd         447
dtype: int32